# arc3-serving-lab3 — scored-GPU throughput RE-MEASURE (NO games, NO submission)

**Why:** five consecutive live reads fell monotonically (1.66 → 1.50 → 1.24 →
1.00 → 0.99) across DIFFERENT arms, including two draws of **byte-identical**
pack-v22 code (1.66 then 0.99). Same-bytes sigma is ~0.19, so a 0.67 spread is
~p0.01, and so is a 5-long monotone run. All runs completed cleanly. This
kernel tests the one environmental explanation we can measure directly:
**did the scored GPU pool's throughput degrade?**

**Method:** boot the exact scored serve chain (anim bundle setup, Qwen3.8-27B
FP8 repack, prefix caching ON, KV bf16 — no deviations) and re-run the SAME
duck-shaped load generator (17-25k-token prompts carrying board images,
thinking ON, temp 0.6 / top_p 0.95 / top_k 20, 1-3k gen tokens) at conc 8 and
conc 28. Nothing else runs.

| Phase | What | Budget |
|---|---|---|
| boot | GPU assert → config → audit → bundle setup (boots vLLM) → weight attestation | ~20 min |
| env | nvidia-smi (name/driver/CUDA/clocks/power/throttle), host CPU+RAM, vLLM version, boot-reported KV cache size + max concurrency, served argv | ~1 min |
| 0 | parser round-trip (tool-call + reasoning parse still healthy) | ~2 min |
| 1 | conc-1 single-stream decode probe (isolates raw GPU speed from batching) | ~3 min |
| 2 | conc-8 duck-shaped load, 90 s warmup + 9 min measure | ~12 min |
| 3 | conc-28 duck-shaped load, 90 s warmup + 10 min measure | ~13 min |
| final | comparison table vs 08-22/08-23 + verdict | ~1 min |

**Reference (arc3-serving-lab v4, 08-22, same GPU pool):** conc8 **1626.0**,
conc16 1140.1, conc28 **642.6** gen-tok/min/session; arc3-serving-lab2 v1
reproduced conc28 at **699.1** (1.09x) on 08-22/23.

**DECISION RULE (pre-registered):** conc-28 per-session gen tok/min
**>= 550 → THROUGHPUT UNCHANGED** (the drift is elsewhere: hidden set,
gateway, or our own arms) · **400-550 → MODERATE degradation** ·
**< 400 → SEVERE degradation explains the score decline**.

Results land in `/kaggle/working/serving_lab3_results.json`, rewritten after
every phase. This kernel never plays a game and never touches the competition
rerun path.


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
# ===================== FAIL-FAST GPU ASSERT (before any setup) ==============
# This lab is only meaningful on the RTX Pro 6000 pool (the scored pool). Die
# IMMEDIATELY on a P100/T4 rehoming so the slot costs minutes, not hours.
_gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True)
GPU_NAME = (_gpu_query.stdout or "").strip()
print(f"serving-lab3: GPU = {GPU_NAME!r} (rc={_gpu_query.returncode})")
if _gpu_query.returncode != 0 or not GPU_NAME:
    raise RuntimeError("WRONG-GPU: nvidia-smi failed — no usable GPU. Aborting fast.")
_gpu_upper = GPU_NAME.upper()
if "6000" not in _gpu_upper or "RTX" not in _gpu_upper:
    raise RuntimeError(
        f"WRONG-GPU: expected an RTX Pro 6000, got {GPU_NAME!r}. Aborting fast "
        "so the session dies in minutes (P100/T4 rehoming).")
print("serving-lab3: GPU assert PASS", flush=True)


In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


def _patch_qwen38_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    Qwen3.8 Kaggle Model. This avoids copying/forking the large bundled setup
    script and keeps the wheelhouse/GPU/vLLM behavior from the source bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
    }

    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for Qwen3.8. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: Qwen3.8 setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_qwen38_setup_commands(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use Qwen3.8.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {QWEN_SERVED_MODEL_NAME!r}"
    )

print("\n✅ TAAF/vLLM setup completed for Qwen3.8")
print("Model path:", QWEN_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
# Boot attestation (doctrine v2, 2026-08-17): the mounted weights must be the
# OFFICIAL Qwen3.8-FP8. Discriminators verified offline against both the
# official HF snapshot and the vrfai 3.6 config: 3.8 = quant_method fp8 /
# fmt e4m3 / transformers 5.8.0.dev0; 3.6-vrfai = compressed-tensors
# config_groups / transformers 5.6.2. A wrong mount must DIE here, before any
# game action is spent. --served-model-name is a rename and proves nothing.
import hashlib as _hashlib
import urllib.request as _rq

_cfg_path = QWEN_MODEL_PATH / "config.json"
_cfg_raw = _cfg_path.read_bytes()
_cfg = json.loads(_cfg_raw)
_q = _cfg.get("quantization_config") or {}
assert _cfg.get("architectures") == ["Qwen3_5ForConditionalGeneration"], (
    f"attest FAIL: architectures {_cfg.get('architectures')}")
assert _q.get("quant_method") == "fp8" and _q.get("fmt") == "e4m3", (
    f"attest FAIL: quantization_config is not official fp8/e4m3: {_q}")
assert _cfg.get("transformers_version") == "5.8.0.dev0", (
    f"attest FAIL: transformers_version {_cfg.get('transformers_version')} "
    "(vrfai 3.6 stamps 5.6.2)")
print("attest: config sha256", _hashlib.sha256(_cfg_raw).hexdigest())

_idx_path = QWEN_MODEL_PATH / "model.safetensors.index.json"
if _idx_path.is_file():
    print("attest: index sha256", _hashlib.sha256(_idx_path.read_bytes()).hexdigest())
_shards = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
assert _shards, "attest FAIL: no safetensors shards at model path"
_total = sum(p.stat().st_size for p in _shards)
print(f"attest: {len(_shards)} shards, {_total} bytes total")
assert _total > 25_000_000_000, f"attest FAIL: total shard bytes {_total} too small for 27B FP8"
_h = _hashlib.sha256()
with open(_shards[0], "rb") as _f:
    _h.update(_f.read(1 << 20))
print("attest: first-shard-1MiB sha256", _h.hexdigest())

# Greedy decode fingerprint — logged (not asserted) for cross-run comparison.
_base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "http://127.0.0.1:1234/v1").rstrip("/")
if not _base.endswith("/v1"):
    _base += "/v1"
_body = json.dumps({
    "model": QWEN_SERVED_MODEL_NAME,
    "messages": [{"role": "user", "content": "Reply with exactly the sum of 17 and 25, then the word quack."}],
    "temperature": 0.0,
    "max_tokens": 48,
    "chat_template_kwargs": {"enable_thinking": False},
}).encode()
_req = _rq.Request(_base + "/chat/completions", data=_body, headers={
    "Content-Type": "application/json",
    "Authorization": "Bearer " + (os.environ.get("LOCAL_ANALYZER_API_KEY") or "EMPTY"),
})
with _rq.urlopen(_req, timeout=180) as _resp:
    _reply = json.loads(_resp.read())["choices"][0]["message"].get("content") or ""
print("attest: decode fingerprint", repr(_reply)[:160])
print("attest: decode sha256", _hashlib.sha256(_reply.encode()).hexdigest())
print("attest: OK — official Qwen3.8-FP8 signature verified before any game")


In [ ]:
# ============================ SERVING LAB LIBRARY ============================
# No games are played in this kernel. Two questions, one commit:
#   ONE question: is the scored GPU pool still delivering the 08-22
#   throughput (1626 @conc8 / 642.6 @conc28 gen-tok/min/session)?
#   The load generator below is byte-identical to the 08-22 run.
import base64
import io
import random
import re
import statistics
import threading
import traceback
import urllib.error
import urllib.request

VLLM_HOST = "127.0.0.1"
VLLM_PORT = 1234
VLLM_ROOT = f"http://{VLLM_HOST}:{VLLM_PORT}"
VLLM_API = VLLM_ROOT + "/v1"
VLLM_MAX_MODEL_LEN = 65536
SITE_PACKAGES = WORKING_DIR / "vllm-site-packages"
RESULTS_PATH = WORKING_DIR / "serving_lab3_results.json"

LAB_HARD_CAP_MIN = 80.0    # trimmed lab: skip any phase starting after this
NST2_GATE_MIN = 110.0      # nst=2 arm only if reached before this
MODAL_H100_TOKMIN = 445.0  # measured gen-tok/min/session at conc 28 (reference)

Q1_RULE = (">=550 tok/min/session at conc 28 = throughput UNCHANGED; "
           "400-550 = MODERATE degradation; <400 = SEVERE degradation")

# Exact scored serve flags (June bundle setup_commands.json, md5 99e4b35d...),
# minus the prefix-caching flag which is parameterized per phase. KV cache
# stays DEFAULT bf16 — we deliberately do NOT copy keithtyser's
# --kv-cache-dtype fp8 (no calibrated KV scales; corruption reports #42179).
BASE_SERVE_FLAGS = [
    "--model", str(QWEN_MODEL_PATH),
    "--served-model-name", QWEN_SERVED_MODEL_NAME,
    "--host", VLLM_HOST,
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "1",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--generation-config", "vllm",
    "--default-chat-template-kwargs", '{"preserve_thinking": true}',
    "--reasoning-parser", "qwen3",
    "--max-model-len", str(VLLM_MAX_MODEL_LEN),
]

CURRENT_SERVER = {"proc": None,
                  "log": str(WORKING_DIR / "vllm-openai-server.log"),
                  "tag": "baseline-scored-flags"}

RESULTS = {
    "meta": {
        "kernel": "arc3-serving-lab3",
        "started_utc": datetime.utcnow().isoformat() + "Z",
        "model_path": str(QWEN_MODEL_PATH),
        "served_model_name": QWEN_SERVED_MODEL_NAME,
        "max_model_len": VLLM_MAX_MODEL_LEN,
        "q1_rule": Q1_RULE,
        "modal_h100_tokmin_session_conc28": MODAL_H100_TOKMIN,
        "baseline_flags": BASE_SERVE_FLAGS + ["--enable-prefix-caching"],
        "kv_cache_dtype": "default (bf16) — fp8 KV deliberately NOT copied",
        "spec_metric_names_verified": [
            "vllm:spec_decode_num_drafts",
            "vllm:spec_decode_num_draft_tokens",
            "vllm:spec_decode_num_accepted_tokens",
            "vllm:spec_decode_num_accepted_tokens_per_pos",
        ],
    },
    "phases": {},
    "verdicts": {},
}


def elapsed_min():
    return (time.time() - NOTEBOOK_START_EPOCH) / 60.0


def save_results():
    tmp = RESULTS_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(RESULTS, indent=2, default=str) + "\n", encoding="utf-8")
    tmp.replace(RESULTS_PATH)


def http_json(url, payload=None, timeout=120):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def http_text(url, timeout=20):
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        return resp.read().decode("utf-8", errors="replace")


def server_alive(timeout=5):
    try:
        http_json(VLLM_API + "/models", timeout=timeout)
        return True
    except Exception:
        return False


def vllm_procs():
    out = subprocess.run(["pgrep", "-f", "vllm.entrypoints"], capture_output=True, text=True)
    return [int(x) for x in out.stdout.split() if x.strip().isdigit()]


def gpu_sample():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=20)
        util, mem = out.stdout.strip().splitlines()[0].split(",")
        return {"util_pct": int(util.strip()), "mem_mib": int(mem.strip())}
    except Exception:
        return None


def tail_log_lines(path, max_bytes=524288):
    p = Path(path)
    if not p.exists():
        return []
    with p.open("rb") as handle:
        handle.seek(0, 2)
        size = handle.tell()
        handle.seek(max(0, size - max_bytes))
        return handle.read().decode("utf-8", errors="replace").splitlines()


def tail_log(path, n=60):
    return "\n".join(tail_log_lines(path)[-n:])


# ---- Prometheus scrape (metric names verified at vLLM v0.19.0 tag) ----------
METRIC_RE = re.compile(r"^(vllm:[A-Za-z0-9_]+)(?:\{[^}]*\})?\s+([0-9.eE+-]+|NaN|nan)\s*$")
WANTED_PREFIXES = ("vllm:spec_decode", "vllm:generation_tokens",
                   "vllm:prompt_tokens", "vllm:prefix_cache",
                   "vllm:num_preemptions", "vllm:num_requests")


def scrape_metrics():
    try:
        text = http_text(VLLM_ROOT + "/metrics", timeout=25)
    except Exception as exc:
        return {}, [f"scrape-failed: {exc!r}"]
    counters, sample_lines = {}, []
    for line in text.splitlines():
        if line.startswith("#"):
            continue
        base = line.split("{")[0].split(" ")[0]
        if not base.startswith(WANTED_PREFIXES):
            continue
        if len(sample_lines) < 40:
            sample_lines.append(line)
        match = METRIC_RE.match(line)
        if not match:
            continue
        name = match.group(1)
        if name.endswith("_total"):
            name = name[:-len("_total")]
        try:
            counters[name] = counters.get(name, 0.0) + float(match.group(2))
        except ValueError:
            pass
    return counters, sample_lines


def acceptance_delta(before, after):
    def delta(name):
        return after.get(name, 0.0) - before.get(name, 0.0)
    drafts = delta("vllm:spec_decode_num_drafts")
    draft_toks = delta("vllm:spec_decode_num_draft_tokens")
    accepted = delta("vllm:spec_decode_num_accepted_tokens")
    return {
        "num_drafts": drafts,
        "num_draft_tokens": draft_toks,
        "num_accepted_tokens": accepted,
        "acceptance_rate": (accepted / draft_toks) if draft_toks > 0 else None,
        "mean_accepted_per_draft": (accepted / drafts) if drafts > 0 else None,
    }


def running_requests():
    counters, _ = scrape_metrics()
    return counters.get("vllm:num_requests_running")


def drain_inflight(max_wait_s=180):
    # After stop, in-flight requests keep decoding; wait so they don't pollute
    # the next phase's measurement window.
    deadline = time.time() + max_wait_s
    while time.time() < deadline:
        active = running_requests()
        if active is None or active <= 0:
            break
        time.sleep(10)


def log_acceptance_lines(n=5):
    # vLLM v0.19.0 vllm/v1/spec_decode/metrics.py logs "Avg Draft acceptance rate: %.1f%%"
    return [ln for ln in tail_log_lines(CURRENT_SERVER["log"])
            if "acceptance rate" in ln][-n:]


# ---- server lifecycle -------------------------------------------------------
def stop_server(reason):
    print(f"serving-lab3: stopping vLLM ({reason})", flush=True)
    subprocess.run(["pkill", "-TERM", "-f", "vllm.entrypoints"], check=False)
    deadline = time.time() + 90
    while time.time() < deadline and vllm_procs():
        time.sleep(3)
    if vllm_procs():
        subprocess.run(["pkill", "-9", "-f", "vllm.entrypoints"], check=False)
        time.sleep(10)
    deadline = time.time() + 240
    while time.time() < deadline:
        sample = gpu_sample()
        if sample is not None and sample["mem_mib"] < 8000:
            break
        time.sleep(5)
    print(f"serving-lab3: server stopped, gpu={gpu_sample()}", flush=True)


def start_server(extra_flags, tag, timeout_s=1500):
    log_path = WORKING_DIR / f"vllm-{tag}.log"
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           *BASE_SERVE_FLAGS, *extra_flags]
    env = os.environ.copy()
    pypath = env.get("PYTHONPATH", "")
    if str(SITE_PACKAGES) not in pypath.split(os.pathsep):
        env["PYTHONPATH"] = f"{SITE_PACKAGES}{os.pathsep}{pypath}" if pypath else str(SITE_PACKAGES)
    env.update({"USE_TF": "0", "TRANSFORMERS_NO_TF": "1",
                "TRANSFORMERS_NO_TORCHVISION": "1", "VLLM_NO_USAGE_STATS": "1",
                "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"})
    print("serving-lab3: starting vLLM:", " ".join(cmd), flush=True)
    handle = log_path.open("w", encoding="utf-8")
    proc = subprocess.Popen(cmd, env=env, stdout=handle, stderr=subprocess.STDOUT, text=True)
    CURRENT_SERVER.update({"proc": proc, "log": str(log_path), "tag": tag})
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(
                f"vLLM ({tag}) died during startup rc={proc.returncode}\n" + tail_log(log_path))
        if server_alive():
            print(f"serving-lab3: vLLM ready ({tag})", flush=True)
            return
        time.sleep(5)
    raise TimeoutError(f"vLLM ({tag}) not ready in {timeout_s}s\n" + tail_log(log_path))


# ---- duck-shaped request synthesis -----------------------------------------
PALETTE = [(0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64),
           (255, 220, 0), (170, 170, 170), (240, 18, 190), (255, 133, 27),
           (128, 219, 255), (135, 12, 37), (105, 58, 183), (63, 81, 181),
           (255, 255, 255)]


def _png_rgb(rows):
    # Minimal pure-stdlib PNG encoder (fallback if PIL is unavailable).
    import struct
    import zlib
    height = len(rows)
    width = len(rows[0])
    raw = b"".join(b"\x00" + b"".join(bytes(px) for px in row) for row in rows)

    def chunk(tag, data):
        body = tag + data
        return struct.pack(">I", len(data)) + body + struct.pack(">I", zlib.crc32(body) & 0xFFFFFFFF)

    ihdr = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", ihdr)
            + chunk(b"IDAT", zlib.compress(raw, 6)) + chunk(b"IEND", b""))


def board_png_b64(rng):
    # 64x64 board upscaled 4x -> 256x256 (matches MULTIMODAL_UPSCALE=4).
    cells, scale = 64, 4
    base = rng.randrange(len(PALETTE))
    grid = [[PALETTE[rng.randrange(len(PALETTE))] if rng.random() < 0.15
             else PALETTE[(base + x // 8 + y // 8) % len(PALETTE)]
             for x in range(cells)] for y in range(cells)]
    try:
        from PIL import Image
        img = Image.new("RGB", (cells, cells))
        for y in range(cells):
            for x in range(cells):
                img.putpixel((x, y), grid[y][x])
        img = img.resize((cells * scale, cells * scale), Image.NEAREST)
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        raw = buf.getvalue()
    except Exception:
        rows = []
        for y in range(cells):
            row = []
            for x in range(cells):
                row.extend([grid[y][x]] * scale)
            rows.extend(list(row) for _ in range(scale))
        raw = _png_rgb(rows)
    return base64.b64encode(raw).decode("ascii")


WORDS = ("grid cluster border sprite agent portal key door wall floor toggle "
         "rotate mirror count color region path move click reward level frame "
         "delta pixel row col mask object pattern rule hypothesis verify plan "
         "act observe anchor cursor palette symmetry adjacency corridor").split()


def make_transcript(rng, approx_tokens):
    lines, tokens, step = [], 0, 0
    while tokens < approx_tokens:
        step += 1
        words = " ".join(rng.choice(WORDS) for _ in range(24))
        lines.append(f"[turn {step:04d}] obs: {words}. delta_pixels="
                     f"{rng.randrange(900)} score={rng.randrange(7)}")
        tokens += 34
    return "\n".join(lines)


SYSTEM_TEXT = ("You are an ARC-AGI-3 game-playing analyst. Maintain a world "
               "model, goal model and action model from board observations, "
               "then choose the next batch of actions. Think carefully.\n"
               + make_transcript(random.Random(7), 1800))

PYTHON_TOOL = [{
    "type": "function",
    "function": {
        "name": "python",
        "description": ("Run Python code against the current game state. The "
                        "snippet is ephemeral and is not saved across calls."),
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string",
                                    "description": "Python code to run."}},
            "required": ["code"],
        },
    },
}]


def _strip_images(user_msg):
    content = [part for part in user_msg["content"] if part.get("type") != "image_url"]
    return {"role": "user", "content": content}


class Session:
    # One synthetic duck "game worker": a growing conversation whose prefix is
    # stable across turns (what makes prefix caching realistic in Phase A).
    def __init__(self, idx, seed, images_per_req=5):
        self.rng = random.Random(seed)
        self.idx = idx
        self.images_per_req = images_per_req
        self.turns = []
        self.base_context = make_transcript(self.rng, 11000 + self.rng.randrange(4000))
        self.last_prompt_tokens = None
        self._pending_user = None

    def _new_user_turn(self):
        text = ("[turn] board updated; analyze the change and choose the next "
                "actions.\n" + make_transcript(self.rng, 400))
        content = [{"type": "text", "text": text},
                   {"type": "image_url", "image_url": {
                       "url": "data:image/png;base64," + board_png_b64(self.rng)}}]
        return {"role": "user", "content": content}

    def build_messages(self):
        msgs = [{"role": "system", "content": SYSTEM_TEXT},
                {"role": "user", "content": [{"type": "text", "text":
                    "Game transcript so far:\n" + self.base_context}]},
                {"role": "assistant", "content": "Understood. World model initialized."}]
        total = len(self.turns)
        for i, (user_msg, assistant_msg) in enumerate(self.turns):
            if total - i > self.images_per_req:
                user_msg = _strip_images(user_msg)
            msgs.append(user_msg)
            msgs.append(assistant_msg)
        self._pending_user = self._new_user_turn()
        msgs.append(self._pending_user)
        return msgs

    def record(self, assistant_text, usage):
        reply = (assistant_text or "").strip()[:1500] or "(thinking only)"
        self.turns.append((self._pending_user, {"role": "assistant", "content": reply}))
        tokens = usage.get("prompt_tokens")
        self.last_prompt_tokens = tokens
        while tokens and tokens > 24500 and len(self.turns) > 2:
            self.turns.pop(0)
            tokens -= 900


def chat_request(session, max_tokens, timeout=1500):
    payload = {
        "model": QWEN_SERVED_MODEL_NAME,
        "messages": session.build_messages(),
        "temperature": 0.6,
        "top_p": 0.95,
        "top_k": 20,
        "max_tokens": max_tokens,
        "chat_template_kwargs": {"enable_thinking": True},
    }
    started = time.time()
    resp = http_json(VLLM_API + "/chat/completions", payload, timeout=timeout)
    latency = time.time() - started
    usage = resp.get("usage") or {}
    message = resp["choices"][0]["message"]
    session.record(message.get("content") or "", usage)
    return {"t_end": time.time(), "latency_s": latency,
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0),
            "had_reasoning": bool(message.get("reasoning_content"))}


# ---- load phase -------------------------------------------------------------
def per_session_of(phase):
    if not phase:
        return None
    metric = phase.get("gen_tok_min_session_metric")
    return metric if metric is not None else phase.get("gen_tok_min_session_usage")


def run_load_phase(name, conc, warmup_s, measure_s):
    if elapsed_min() > LAB_HARD_CAP_MIN:
        print(f"serving-lab3: SKIP {name} — past hard cap ({elapsed_min():.0f} min)", flush=True)
        RESULTS["phases"][name] = {"skipped": "hard-cap"}
        save_results()
        return None
    if not server_alive(15):
        print(f"serving-lab3: SKIP {name} — server not alive", flush=True)
        RESULTS["phases"][name] = {"skipped": "server-dead"}
        save_results()
        return None
    print(f"\nserving-lab3: === {name} (conc={conc}, warmup={warmup_s}s, "
          f"measure={measure_s}s, elapsed={elapsed_min():.1f} min) ===", flush=True)
    events, errors = [], []
    lock = threading.Lock()
    stop = threading.Event()
    gpu_samples = []

    def sampler():
        while not stop.is_set():
            sample = gpu_sample()
            if sample:
                gpu_samples.append(sample)
            stop.wait(30)

    def worker(i):
        session = Session(i, seed=(hash(name) & 0xFFFF) * 100 + i)
        while not stop.is_set():
            max_tok = session.rng.choice((1024, 2048, 3072))
            try:
                record = chat_request(session, max_tok)
                with lock:
                    events.append(record)
            except urllib.error.HTTPError as exc:
                try:
                    body = exc.read().decode("utf-8", errors="replace")[:400]
                except Exception:
                    body = ""
                with lock:
                    errors.append({"t": time.time(), "code": exc.code, "body": body})
                if "image" in body.lower() or "multi" in body.lower():
                    session.images_per_req = 1
                stop.wait(3)
            except Exception as exc:
                with lock:
                    errors.append({"t": time.time(), "err": repr(exc)[:200]})
                stop.wait(5)

    threads = [threading.Thread(target=worker, args=(i,), daemon=True) for i in range(conc)]
    threads.append(threading.Thread(target=sampler, daemon=True))
    for thread in threads:
        thread.start()
    time.sleep(warmup_s)
    metrics_before, _ = scrape_metrics()
    t0 = time.time()
    server_died = False
    while time.time() - t0 < measure_s:
        time.sleep(15)
        if not server_alive(10) and not vllm_procs():
            server_died = True
            print(f"serving-lab3: SERVER DIED during {name}", flush=True)
            break
    t1 = time.time()
    metrics_after, metric_lines = scrape_metrics()
    stop.set()
    for thread in threads:
        thread.join(timeout=2)
    if not server_died:
        drain_inflight()

    window = [e for e in events if t0 <= e["t_end"] <= t1]
    minutes = max((t1 - t0) / 60.0, 0.01)
    usage_gen = sum(e["completion_tokens"] for e in window)
    gen_delta = metrics_after.get("vllm:generation_tokens", 0.0) - \
        metrics_before.get("vllm:generation_tokens", 0.0)
    latencies = sorted(e["latency_s"] for e in window)
    prompts = [e["prompt_tokens"] for e in window]
    result = {
        "conc": conc,
        "warmup_s": warmup_s,
        "measure_min": round(minutes, 2),
        "requests_in_window": len(window),
        "requests_total": len(events),
        "errors": len(errors),
        "error_samples": errors[:5],
        "gen_tok_min_session_metric": round(gen_delta / minutes / conc, 1) if gen_delta > 0 else None,
        "gen_tok_min_session_usage": round(usage_gen / minutes / conc, 1),
        "gen_tok_s_aggregate_metric": round(gen_delta / (minutes * 60.0), 1) if gen_delta > 0 else None,
        "gen_tok_s_aggregate_usage": round(usage_gen / (minutes * 60.0), 1),
        "mean_latency_s": round(statistics.fmean(latencies), 1) if latencies else None,
        "p50_latency_s": round(statistics.median(latencies), 1) if latencies else None,
        "prompt_tokens_mean": round(statistics.fmean(prompts)) if prompts else None,
        "prompt_tokens_min": min(prompts) if prompts else None,
        "prompt_tokens_max": max(prompts) if prompts else None,
        "completion_tokens_mean": round(statistics.fmean(
            [e["completion_tokens"] for e in window])) if window else None,
        "acceptance": acceptance_delta(metrics_before, metrics_after),
        "acceptance_log_lines": log_acceptance_lines(),
        "spec_metric_lines_sample": metric_lines[:8],
        "gpu_samples_tail": gpu_samples[-6:],
        "server_died": server_died,
        "server_alive_at_end": server_alive(),
        "server_tag": CURRENT_SERVER["tag"],
    }
    RESULTS["phases"][name] = result
    save_results()
    print(f"serving-lab3: {name}: {result['gen_tok_min_session_metric'] or result['gen_tok_min_session_usage']}"
          f" gen-tok/min/session ({len(window)} reqs, {len(errors)} errs, "
          f"prompt~{result['prompt_tokens_mean']}, p50 lat {result['p50_latency_s']}s)", flush=True)
    if result["acceptance"]["num_draft_tokens"] > 0:
        print(f"serving-lab3: {name}: acceptance_rate="
              f"{result['acceptance']['acceptance_rate']:.3f} "
              f"mean_accepted_per_draft={result['acceptance']['mean_accepted_per_draft']:.2f}",
              flush=True)
    return result


# ---- parser round-trip ------------------------------------------------------
def parser_roundtrip(tag):
    name = f"parser_roundtrip_{tag}"
    if not server_alive(15):
        RESULTS["phases"][name] = {"skipped": "server-dead"}
        save_results()
        return
    prompts = [
        ("auto", "The board has an unknown number of red pixels. Use the python "
                 "tool to inspect: call it with code that prints grid[0][0]."),
        ("auto", "You must act now. Emit a python tool call whose code prints "
                 "the string 'quack' and nothing else."),
        ("forced", "Count from 1 to 3 using the python tool."),
    ]
    outcomes = []
    for mode, prompt in prompts:
        payload = {
            "model": QWEN_SERVED_MODEL_NAME,
            "messages": [
                {"role": "system", "content":
                    "You are an ARC-AGI-3 analyst. Use the python tool to act."},
                {"role": "user", "content": prompt},
            ],
            "tools": PYTHON_TOOL,
            "temperature": 0.6,
            "top_p": 0.95,
            "top_k": 20,
            "max_tokens": 1200,
            "chat_template_kwargs": {"enable_thinking": True},
        }
        if mode == "forced":
            payload["tool_choice"] = {"type": "function", "function": {"name": "python"}}
        try:
            resp = http_json(VLLM_API + "/chat/completions", payload, timeout=600)
            message = resp["choices"][0]["message"]
            tool_calls = message.get("tool_calls") or []
            ok, detail = False, ""
            if tool_calls:
                fn = tool_calls[0].get("function", {})
                args_ok = False
                try:
                    args = json.loads(fn.get("arguments") or "{}")
                    args_ok = isinstance(args.get("code"), str)
                except Exception:
                    args = None
                ok = fn.get("name") == "python" and args_ok
                detail = f"name={fn.get('name')} args_parse={'OK' if args_ok else 'FAIL'}"
            else:
                detail = (f"no tool_calls; finish={resp['choices'][0].get('finish_reason')}; "
                          f"content_head={(message.get('content') or '')[:80]!r}")
            outcomes.append({"mode": mode, "ok": ok, "detail": detail})
        except Exception as exc:
            outcomes.append({"mode": mode, "ok": False, "detail": repr(exc)[:250]})
    ok_count = sum(1 for o in outcomes if o["ok"])
    verdict = "PASS" if ok_count >= 2 else "FAIL"
    RESULTS["phases"][name] = {"ok_count": ok_count, "total": len(outcomes),
                               "verdict": verdict, "outcomes": outcomes}
    save_results()
    print(f"serving-lab3: {name}: {verdict} ({ok_count}/{len(outcomes)} tool calls parsed)", flush=True)


# ---- long-context soak (FP8+MTP crash hunt, vllm #40756) --------------------
CRASH_PATTERNS = ("illegal memory access", "CUDA error", "device-side assert",
                  "core dumped", "Engine core initialization failed",
                  "Watchdog caught")


def long_context_soak(tag, conc=6, duration_s=480, target_prompt_tokens=26000):
    name = f"{tag}_soak_longctx"
    if elapsed_min() > LAB_HARD_CAP_MIN:
        RESULTS["phases"][name] = {"skipped": "hard-cap"}
        save_results()
        return
    if not server_alive(15):
        RESULTS["phases"][name] = {"skipped": "server-dead-before-soak",
                                   "verdict": "MTP-FATAL (server already dead before soak)"}
        save_results()
        print("serving-lab3: MTP-FATAL — server dead before long-context soak", flush=True)
        return
    print(f"\nserving-lab3: === {name} (conc={conc}, {duration_s}s, "
          f"~{target_prompt_tokens}-token prompts) ===", flush=True)
    events, errors = [], []
    lock = threading.Lock()
    stop = threading.Event()
    counter = {"n": 0}

    def worker(i):
        while not stop.is_set():
            with lock:
                counter["n"] += 1
                seq = counter["n"]
            rng = random.Random(90000 + seq)
            text = make_transcript(rng, target_prompt_tokens)
            messages = [
                {"role": "system", "content": SYSTEM_TEXT},
                {"role": "user", "content": [
                    {"type": "text", "text": "Full game transcript:\n" + text
                        + "\nAnalyze deeply and produce a plan."},
                    {"type": "image_url", "image_url": {
                        "url": "data:image/png;base64," + board_png_b64(rng)}},
                ]},
            ]
            payload = {"model": QWEN_SERVED_MODEL_NAME, "messages": messages,
                       "temperature": 0.6, "top_p": 0.95, "top_k": 20,
                       "max_tokens": 2048,
                       "chat_template_kwargs": {"enable_thinking": True}}
            started = time.time()
            try:
                resp = http_json(VLLM_API + "/chat/completions", payload, timeout=1500)
                usage = resp.get("usage") or {}
                with lock:
                    events.append({"latency_s": round(time.time() - started, 1),
                                   "prompt_tokens": usage.get("prompt_tokens", 0),
                                   "completion_tokens": usage.get("completion_tokens", 0)})
            except Exception as exc:
                with lock:
                    errors.append({"t": time.time(), "err": repr(exc)[:200]})
                stop.wait(5)

    threads = [threading.Thread(target=worker, args=(i,), daemon=True) for i in range(conc)]
    for thread in threads:
        thread.start()
    t0 = time.time()
    server_died = False
    while time.time() - t0 < duration_s:
        time.sleep(15)
        if not server_alive(10) and not vllm_procs():
            server_died = True
            print(f"serving-lab3: SERVER DIED during {name}", flush=True)
            break
    stop.set()
    for thread in threads:
        thread.join(timeout=2)
    crash_lines = [ln for ln in tail_log_lines(CURRENT_SERVER["log"])
                   if any(pat in ln for pat in CRASH_PATTERNS)][:10]
    alive = server_alive()
    if server_died or (not alive and not vllm_procs()):
        verdict = "MTP-FATAL (server died during long-context soak — vllm #40756 class)"
    elif crash_lines:
        verdict = "SUSPECT (crash strings in server log, server still alive)"
    else:
        verdict = "SURVIVED"
    result = {"conc": conc, "duration_s": duration_s,
              "requests_ok": len(events), "errors": len(errors),
              "error_samples": errors[:5],
              "prompt_tokens_max": max((e["prompt_tokens"] for e in events), default=None),
              "mean_latency_s": round(statistics.fmean(
                  [e["latency_s"] for e in events]), 1) if events else None,
              "server_alive_at_end": alive, "crash_lines": crash_lines,
              "verdict": verdict}
    RESULTS["phases"][name] = result
    save_results()
    print(f"serving-lab3: {name}: {verdict} ({len(events)} ok, {len(errors)} errs, "
          f"max prompt {result['prompt_tokens_max']})", flush=True)


print(f"serving-lab3: library ready, elapsed {elapsed_min():.1f} min")
save_results()


In [ ]:
# ================= ENVIRONMENT CAPTURE (the degradation diff) ===============
# Everything here is a candidate explanation for a throughput change that is
# NOT our code: a different silicon SKU, an older/newer driver, a lower power
# cap or clock ceiling, active throttling, a thinner host CPU/RAM allotment,
# a different vLLM build, or less KV cache (=> less real concurrency).
ENVCAP = {}


def _smi_one(field):
    r = subprocess.run(["nvidia-smi", "--query-gpu=" + field,
                        "--format=csv,noheader"], capture_output=True, text=True)
    out = (r.stdout or "").strip()
    if r.returncode != 0 or not out:
        return None
    return out.splitlines()[0].strip()


def _smi_query(fields):
    # Driver renamed clocks_throttle_reasons.* -> clocks_event_reasons.* ; ONE
    # bad field fails the whole batch query, so fall back field-by-field.
    names = [f.strip() for f in fields.split(",") if f.strip()]
    r = subprocess.run(["nvidia-smi", "--query-gpu=" + ",".join(names),
                        "--format=csv,noheader"], capture_output=True, text=True)
    out = (r.stdout or "").strip()
    if r.returncode == 0 and out:
        vals = [v.strip() for v in out.splitlines()[0].split(",")]
        if len(vals) == len(names):
            return dict(zip(names, vals))
    result = {"_batch_query_failed": (r.stderr or "")[:200]}
    for name in names:
        alt = name.replace("clocks_throttle_reasons", "clocks_event_reasons")
        value = _smi_one(name)
        if value is None and alt != name:
            value = _smi_one(alt)
            if value is not None:
                name = alt
        result[name] = value
    return result


ENVCAP["gpu_static"] = _smi_query(
    "name,driver_version,vbios_version,memory.total,clocks.max.sm,"
    "clocks.max.mem,clocks.max.graphics,power.max_limit,power.limit,"
    "pcie.link.gen.max,pcie.link.width.max,compute_mode,persistence_mode")
ENVCAP["gpu_now"] = _smi_query(
    "clocks.sm,clocks.mem,clocks.graphics,temperature.gpu,power.draw,"
    "utilization.gpu,utilization.memory,memory.used,"
    "clocks_throttle_reasons.active,clocks_throttle_reasons.sw_power_cap,"
    "clocks_throttle_reasons.hw_slowdown,clocks_throttle_reasons.sw_thermal_slowdown")
_smi_all = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
ENVCAP["nvidia_smi_head"] = [ln for ln in (_smi_all.stdout or "").splitlines()[:12]]

try:
    _cpuinfo = Path("/proc/cpuinfo").read_text(errors="replace")
    _models = [ln.split(":", 1)[1].strip() for ln in _cpuinfo.splitlines()
               if ln.lower().startswith("model name")]
    ENVCAP["cpu_model"] = _models[0] if _models else None
    ENVCAP["cpu_logical_count"] = len(_models) or os.cpu_count()
except Exception as _exc:
    ENVCAP["cpu_error"] = repr(_exc)[:200]
ENVCAP["os_cpu_count"] = os.cpu_count()
try:
    _mem = dict(
        (ln.split(":", 1)[0], ln.split(":", 1)[1].strip())
        for ln in Path("/proc/meminfo").read_text().splitlines() if ":" in ln)
    ENVCAP["mem_total"] = _mem.get("MemTotal")
    ENVCAP["mem_available"] = _mem.get("MemAvailable")
except Exception as _exc:
    ENVCAP["mem_error"] = repr(_exc)[:200]
try:
    ENVCAP["loadavg"] = Path("/proc/loadavg").read_text().strip()
    ENVCAP["uptime_s"] = float(Path("/proc/uptime").read_text().split()[0])
except Exception:
    pass

# ---- vLLM identity + what it reported at boot -------------------------------
def _find_boot_log():
    cands = []
    for pat in ("*.log", "**/*.log"):
        for p in WORKING_DIR.glob(pat):
            try:
                if p.stat().st_size > 2000:
                    cands.append(p)
            except OSError:
                pass
    best, best_score = None, -1
    for p in cands:
        try:
            head = p.read_text(errors="replace")[:2_000_000]
        except Exception:
            continue
        score = head.count("vLLM API server version") * 10 + head.count("KV cache")
        if score > best_score:
            best, best_score = p, score
    return best if best_score > 0 else None


_boot_log = _find_boot_log()
ENVCAP["boot_log_path"] = str(_boot_log) if _boot_log else None
_boot_txt = ""
if _boot_log is not None:
    try:
        _boot_txt = _boot_log.read_text(errors="replace")
    except Exception as _exc:
        ENVCAP["boot_log_error"] = repr(_exc)[:200]

_PATTERNS = {
    "vllm_api_server_version": "vLLM API server version",
    "kv_cache_size": "GPU KV cache size",
    "max_concurrency": "Maximum concurrency for",
    "kv_cache_memory": "Available KV cache memory",
    "model_weights_mem": "Model loading took",
    "graph_capture": "graph capturing finished",
    "torch_compile": "torch.compile takes",
    "args_namespace": "args: Namespace",
}
ENVCAP["boot_lines"] = {}
for _key, _needle in _PATTERNS.items():
    _hits = [ln.strip()[:600] for ln in _boot_txt.splitlines() if _needle in ln]
    if _hits:
        ENVCAP["boot_lines"][_key] = _hits[:3]

_ver = subprocess.run(
    [sys.executable, "-c",
     "import vllm, torch, sys; "
     "print(vllm.__version__); print(torch.__version__); "
     "print(torch.version.cuda); print(torch.cuda.get_device_name(0)); "
     "print(torch.cuda.get_device_capability(0))"],
    capture_output=True, text=True, env={**os.environ, "VLLM_NO_USAGE_STATS": "1"})
_vlines = (_ver.stdout or "").strip().splitlines()
if len(_vlines) >= 5:
    ENVCAP["vllm_version"] = _vlines[0]
    ENVCAP["torch_version"] = _vlines[1]
    ENVCAP["torch_cuda"] = _vlines[2]
    ENVCAP["torch_device_name"] = _vlines[3]
    ENVCAP["torch_sm"] = _vlines[4]
else:
    ENVCAP["vllm_version_probe_error"] = ((_ver.stderr or "")[-400:] or "no output")

# served argv (proves the flags actually in force, incl. prefix caching)
_ps = subprocess.run(["ps", "-eo", "args"], capture_output=True, text=True)
ENVCAP["vllm_argv"] = [ln.strip()[:900] for ln in (_ps.stdout or "").splitlines()
                       if "vllm" in ln and ("api_server" in ln or "entrypoints" in ln)][:2]

print(json.dumps(ENVCAP, indent=2, default=str)[:6000], flush=True)
RESULTS["meta"]["environment"] = ENVCAP
RESULTS["meta"]["reference_tokmin_session_0822"] = {'8': 1626.0, '16': 1140.1, '28': 642.6}
RESULTS["meta"]["reference_conc28_reproduction_0823"] = 699.1
save_results()


In [ ]:
# ============ THE MEASUREMENT — parser + conc 1 / 8 / 28 duck load ==========
# Identical load generator, identical flags, identical model as the 08-22 run
# that measured 1626.0 @conc8 and 642.6 @conc28 (and 699.1 @conc28 on 08-23).
try:
    parser_roundtrip("scored_stack")
except Exception:
    traceback.print_exc()
    RESULTS["phases"].setdefault("parser_roundtrip_scored_stack", {"error": "see traceback"})
    save_results()

for _name, _conc, _warm, _meas in [("probe_conc1", 1, 30, 150),
                                   ("remeasure_conc8", 8, 90, 540),
                                   ("remeasure_conc28", 28, 90, 600)]:
    try:
        run_load_phase(_name, _conc, _warm, _meas)
    except Exception:
        traceback.print_exc()
        RESULTS["phases"].setdefault(_name, {"error": "see traceback"})
        save_results()
print("serving-lab3: measurement phases done at", round(elapsed_min(), 1), "min", flush=True)
save_results()


In [ ]:
# ============== FINAL — comparison table + pre-registered verdict ===========
def _cell(value, width=13):
    return str(value if value is not None else "-").rjust(width)


_ref = RESULTS["meta"]["reference_tokmin_session_0822"]
_repro = RESULTS["meta"]["reference_conc28_reproduction_0823"]
_now = {}
for _key, _phase in (("8", "remeasure_conc8"), ("28", "remeasure_conc28")):
    _now[_key] = per_session_of(RESULTS["phases"].get(_phase) or {})

print("\n" + "=" * 92)
print("serving-lab3 — SCORED-GPU THROUGHPUT RE-MEASURE (gen tok/min/session)")
print("=" * 92)
print("  conc |     08-22 ref |  08-23 repro |    2026-08-26 |   ratio vs 08-22")
for _key in ("8", "28"):
    _r = _ref.get(_key)
    _p = _repro if _key == "28" else None
    _n = _now.get(_key)
    _ratio = f"{_n / _r:.2f}x" if (_n and _r) else "-"
    print(f"  {_key:>4} | {_cell(_r)} | {_cell(_p, 12)} | {_cell(_n)} |   {_ratio}")

for _key, _phase in (("1", "probe_conc1"), ("8", "remeasure_conc8"), ("28", "remeasure_conc28")):
    _ph = RESULTS["phases"].get(_phase) or {}
    if not _ph or _ph.get("skipped") or _ph.get("error"):
        print(f"\n  conc {_key}: NO DATA ({_ph.get('skipped') or _ph.get('error') or 'phase missing'})")
        continue
    print(f"\n  conc {_key} detail: {per_session_of(_ph)} tok/min/session | "
          f"aggregate {_ph.get('gen_tok_s_aggregate_metric') or _ph.get('gen_tok_s_aggregate_usage')} tok/s | "
          f"{_ph.get('requests_in_window')} reqs / {_ph.get('errors')} errs | "
          f"prompt mean {_ph.get('prompt_tokens_mean')} "
          f"({_ph.get('prompt_tokens_min')}-{_ph.get('prompt_tokens_max')}) | "
          f"gen mean {_ph.get('completion_tokens_mean')} | "
          f"p50 lat {_ph.get('p50_latency_s')}s | mean lat {_ph.get('mean_latency_s')}s")

_rate28 = _now.get("28")
if _rate28 is None:
    _verdict = "NO-DATA — conc-28 phase did not produce a rate; hypothesis UNTESTED"
elif _rate28 >= 550:
    _verdict = (f"THROUGHPUT UNCHANGED ({_rate28} >= 550 tok/min/session at conc 28) — "
                "the GPU-degradation hypothesis is REFUTED; the score drift is "
                "elsewhere (hidden set, gateway, or our own arms)")
elif _rate28 >= 400:
    _verdict = (f"MODERATE DEGRADATION ({_rate28} in 400-550 tok/min/session at conc 28) — "
                "partial token starvation; contributes to but does not fully "
                "explain the 5-draw decline")
else:
    _verdict = (f"SEVERE DEGRADATION ({_rate28} < 400 tok/min/session at conc 28) — "
                "the scored GPU pool got slower; token starvation explains the "
                "score decline")
RESULTS["verdicts"]["throughput_2026_08_26"] = _verdict
RESULTS["verdicts"]["conc8_tokmin_session"] = _now.get("8")
RESULTS["verdicts"]["conc28_tokmin_session"] = _rate28
RESULTS["verdicts"]["conc1_tokmin_session"] = per_session_of(
    RESULTS["phases"].get("probe_conc1") or {})
RESULTS["verdicts"]["decision_rule"] = (
    ">=550 UNCHANGED | 400-550 MODERATE | <400 SEVERE (conc 28, gen tok/min/session)")
RESULTS["meta"]["finished_utc"] = datetime.utcnow().isoformat() + "Z"
RESULTS["meta"]["total_elapsed_min"] = round(elapsed_min(), 1)

_env = RESULTS["meta"].get("environment", {})
print("\n" + "-" * 92)
print("ENVIRONMENT (compare against the 08-22 run):")
print("  gpu       :", _env.get("gpu_static", {}))
print("  gpu now   :", _env.get("gpu_now", {}))
print("  host      :", _env.get("cpu_model"), "|", _env.get("cpu_logical_count"),
      "vcpu |", _env.get("mem_total"), "| load", _env.get("loadavg"))
print("  vllm      :", _env.get("vllm_version"), "| torch", _env.get("torch_version"),
      "| cuda", _env.get("torch_cuda"), "| sm", _env.get("torch_sm"))
for _k, _v in (_env.get("boot_lines") or {}).items():
    print(f"  boot.{_k}: {_v[0] if _v else ''}")
print("-" * 92)
print("DECISION RULE:", RESULTS["verdicts"]["decision_rule"])
print("VERDICT      :", _verdict)
print("=" * 92, flush=True)
save_results()
print("results ->", RESULTS_PATH, RESULTS_PATH.stat().st_size, "bytes", flush=True)
